# Create positions from tissue boundaries

Builds the FOV scanning positions for one or more tissue sections on a coverslip,
from `boundary_positions*.txt`/`hole*.txt` files in
`positions/boundaries/{manual,from_mosaic}/`. Those files can come from either:
drawing them by hand (put them in `positions/boundaries/manual/`), or running
`02_create_boundary_from_mosaic.ipynb` first to derive them automatically from a
Steve low-mag mosaic (writes to `positions/boundaries/from_mosaic/`) — both feed
this notebook identically; `BOUNDARY_SOURCE` below picks which one to read.

**Tissue count** (auto-detected from the filenames in the resolved boundary folder):
- **multi**  — `tissue_{t}_boundary_positions_{b}.txt`: several tissue sections,
  each possibly split across several boundary files.
- **single** — `boundary_positions_{b}.txt`: one tissue, several boundaries.
- **legacy** — a lone `boundary_positions.txt`: one boundary (original behaviour).

**Per-tissue path mode** (`TISSUE_PATH_MODE` / `TISSUE_PATH_MODE_OVERRIDES`,
independent of the tissue count above): how a tissue's OWN boundaries — if it
has more than one — connect to each other:
- **legacy** (default, for every tissue) — the boundaries are concatenated
  into one continuous path, closed back near its own start; no dedicated
  transit FOVs. Use when a tissue's boundaries are close/contiguous enough
  that the stage can move directly between them.
- **transit** — a dedicated transit segment bridges each consecutive pair of
  that tissue's own boundaries (the original multi-boundary behaviour).

Boundaries belonging to **different tissues** always get a transit bridge —
they are physically separate coverslip regions — regardless of the
per-tissue path mode above, which only governs a tissue's own internal
boundaries.

**Transit FOVs.** Where inserted (between different tissues, always; within
a "transit"-mode tissue's own boundaries), FOVs sit on the straight line from
the last FOV of one segment to the first FOV of the next, spaced ~2× the
grid step, wrapping the last segment back to the first so the whole
acquisition is one cyclic sequence. Transit FOVs are imaged with blank
frames at the bead focus (the `hal-config-*-transit-*` from notebook 01).

**Outputs** (written to `positions/`, with `SAMPLE = POSITIONS_TAG`, i.e.
`SAMPLE_NAME` — the TRUE top-level experiment id auto-detected via
`resolve_sample_identity` — with `_{IMAGING_DIR}` appended in a split layout,
e.g. `LT058_sample_07_merfish`, so a sibling acquisition of the same sample
(e.g. `lineage_tracing/merfish` vs `lineage_tracing/lineage`) never writes an
identically-named positions file. NOT `SAMPLE_DIR.name`, which is only this
acquisition's own local folder name once split into sibling acquisition-type
subfolders):
- per-segment files referenced by Dave: `positions_{SAMPLE}_{label}.txt`
  (`label` = `T{t}B{b}` / `B{b}` / `T{t}`, or omitted for a single combined
  tissue segment) and `positions_{SAMPLE}_transit_{k}.txt`
- per-tissue FOV-only files (no transit): `positions_{SAMPLE}_T{t}.txt`
  (multi) or `positions_{SAMPLE}.txt` (single/legacy)

It also creates the `data/` subfolders for the layout (`mosaic10x`, and either
`tissue_{t}/{cells,hybs,transit}` or top-level `{cells,hybs,transit}`).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.io               import save_positions_array
from MERci.common.experiment_info  import resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs     import get_fov_geometry
from MERci.acquisition.positions   import (
    discover_boundary_files, resolve_boundary_dir, resolve_boundaries_source_dir,
    load_boundary_polygon, load_hole_polygons, group_boundaries_by_path_mode,
    build_boundary_path, create_transit_path, get_path_stats,
)

In [ ]:
POSITIONS_DIR = SAMPLE_DIR / "positions"
DATA_DIR      = SAMPLE_DIR / "data"
POSITIONS_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME is the TRUE top-level experiment id (e.g. "LT058_sample_07"),
# auto-detected from the folder structure -- NOT SAMPLE_DIR.name, which is only
# this acquisition's own local folder name (e.g. "merfish") once split into
# sibling acquisition-type subfolders. IMAGING_DIR is that subfolder name
# ("merfish", "" if flat). POSITIONS_TAG (below) -- not bare SAMPLE_NAME -- is
# what every positions_{...}.txt/fov_layout_{...}.png filename this notebook
# writes actually uses.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

# ── Grid / FOV geometry ──────────────────────────────────────────────────
MICROSCOPE           = "ST2"   # MF2-MF5: 0.108 um/px, 2048 px | MFX/ST2: 0.0878 um/px, 2304 px
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE)  # geometry from the microscope
non_overlap_fraction = 0.9

step_size_um = pixel_size_um * image_size_px * non_overlap_fraction
fov_size_um  = pixel_size_um * image_size_px

# ── Scanning / transit options ───────────────────────────────────────────
SCAN_DIRECTION       = "vertical"   # boustrophedon direction within each boundary
TRANSIT_SPACING      = 2.0          # transit FOV spacing as a multiple of step_size

# Each tissue can have more than one boundary file (e.g. disjoint fragments
# of the same section). How a tissue's OWN boundaries connect to each other
# is independent of how many TISSUES there are (multi/single/legacy below)
# and is chosen here:
#   "legacy"  -- the boundaries are concatenated into one continuous path,
#                closed back near its own start (no dedicated transit FOVs).
#                Appropriate when a tissue's boundaries are close/contiguous
#                enough that the stage can move directly between them.
#   "transit" -- a dedicated transit segment (create_transit_path) bridges
#                each consecutive pair of that tissue's own boundaries.
# Default: "legacy" for every tissue (generalises the original single-
# boundary "legacy" layout to more than one boundary). Override per tissue
# via TISSUE_PATH_MODE_OVERRIDES, e.g. {2: "transit"}. Boundaries belonging
# to DIFFERENT tissues always get a transit bridge regardless of this
# setting (they are physically separate coverslip regions) -- this only
# affects a tissue's OWN internal boundaries.
TISSUE_PATH_MODE           = "legacy"   # "legacy" | "transit" -- default for every tissue
TISSUE_PATH_MODE_OVERRIDES = {}          # e.g. {2: "transit"}

def tissue_path_mode(t):
    mode = TISSUE_PATH_MODE_OVERRIDES.get(t, TISSUE_PATH_MODE)
    if mode not in ("legacy", "transit"):
        raise ValueError(f"Tissue {t}: path mode must be 'legacy' or 'transit', got {mode!r}")
    return mode

# ── Discover the boundary layout ─────────────────────────────────────────
# Boundary inputs live under positions/boundaries/{manual,from_mosaic}/.
# BOUNDARY_SOURCE = None auto-picks whichever of the two has files (preferring
# from_mosaic), so this notebook doesn't need to be told which upstream method
# you used; set it explicitly ("manual" or "from_mosaic") to override.
BOUNDARY_SOURCE = None
SOURCE_DIR, BOUNDARY_SOURCE = resolve_boundaries_source_dir(POSITIONS_DIR, BOUNDARY_SOURCE)

# If the resolved source has no boundary files yet (e.g. a fresh experiment),
# fall back to a bundled EXAMPLE dataset under MERci/data/positions/examples/
# so the notebook can still be run and tested. EXAMPLE_LAYOUT selects which
# example set to use.
EXAMPLE_LAYOUT = "legacy"     # "legacy" | "single" | "multi"
EXAMPLE_ROOT   = MERCI_DIR / "data" / "positions" / "examples"
BOUNDARY_DIR, USING_EXAMPLE = resolve_boundary_dir(SOURCE_DIR, EXAMPLE_ROOT, EXAMPLE_LAYOUT)
if USING_EXAMPLE:
    # SOURCE_DIR has no boundary files: copy the example boundary + hole inputs
    # into it so the experiment folder is self-contained and notebook 06
    # (which reads the same resolved source directory) finds them. Idempotent:
    # once copied, resolve_boundary_dir returns SOURCE_DIR and this branch is
    # skipped.
    import shutil
    SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    copied = [f.name for f in sorted(BOUNDARY_DIR.glob("*.txt"))]
    for fname in copied:
        shutil.copy2(BOUNDARY_DIR / fname, SOURCE_DIR / fname)
    print(f"WARNING: no boundary files in {SOURCE_DIR};")
    print(f"         copied {len(copied)} example input(s) from '{EXAMPLE_LAYOUT}' -> "
          f"boundaries/{BOUNDARY_SOURCE}/: {copied}\n")
    BOUNDARY_DIR = SOURCE_DIR

boundaries, MODE = discover_boundary_files(BOUNDARY_DIR)
tissues          = sorted({b.tissue for b in boundaries})
n_boundaries     = len(boundaries)

print(f"SAMPLE_DIR     : {SAMPLE_DIR}")
print(f"SAMPLE_NAME    : {SAMPLE_NAME}")
print(f"POSITIONS_TAG  : {POSITIONS_TAG}")
print(f"BOUNDARY_SOURCE: {BOUNDARY_SOURCE}")
print(f"BOUNDARY_DIR   : {BOUNDARY_DIR}")
print(f"Step size      : {step_size_um:.1f} um")
print(f"FOV size       : {fov_size_um:.1f} um")
print(f"\nLayout mode   : {MODE}   ({len(tissues)} tissue(s), {n_boundaries} boundary file(s))")
print(f"Boundaries in acquisition order:")
for b in boundaries:
    print(f"  {b.label or '(single)':8s}  <- {b.path.name}")
print(f"\nPer-tissue path mode:")
for t in tissues:
    n_b = sum(1 for b in boundaries if b.tissue == t)
    print(f"  tissue {t}: {tissue_path_mode(t):8s} ({n_b} boundary file(s))")

In [ ]:
# ── Load boundaries + holes, build per-boundary FOV paths ────────────────
# Holes are global: every hole*.txt applies to every boundary.
holes = load_hole_polygons(BOUNDARY_DIR)
print(f"Loaded {len(holes)} hole polygon(s) (applied to all boundaries)")

boundary_polys = [load_boundary_polygon(b.path) for b in boundaries]

# Group consecutive same-tissue boundaries per TISSUE_PATH_MODE -- shared
# with notebook 06's create_round_info_multitissue (both call
# group_boundaries_by_path_mode), so the two can never disagree about which
# positions files this notebook actually writes below.
groups = group_boundaries_by_path_mode(boundaries, MODE, tissue_path_mode)

def _boundary_return_side(spec):
    # Close the loop ("top" side moved to the end -- see close_scanning_path)
    # whenever this boundary's own path stands alone with no dedicated
    # transit FOV to hand off to: either it's the tissue's only boundary, or
    # its tissue is in "legacy" mode (its boundaries are concatenated
    # directly, so each one needs to end near where the next one begins). In
    # "transit" mode the transit segment itself bridges the gap, so the raw
    # (unclosed) snake order is kept, exactly as before.
    n_in_tissue = sum(1 for s in boundaries if s.tissue == spec.tissue)
    if n_in_tissue == 1 or tissue_path_mode(spec.tissue) == "legacy":
        return "top"
    return None

boundary_paths = [
    build_boundary_path(
        poly, holes, step_size_um, fov_size_um,
        direction   = SCAN_DIRECTION,
        return_side = _boundary_return_side(spec),
    )
    for spec, poly in zip(boundaries, boundary_polys)
]
for spec, path in zip(boundaries, boundary_paths):
    if len(path) == 0:
        raise ValueError(
            f"Boundary {spec.label or spec.path.name} produced 0 FOVs — "
            f"check the boundary file and grid geometry."
        )

# "legacy" tissues assume their own boundaries are close/contiguous enough
# that no dedicated transit is needed between them -- warn if that
# assumption looks wrong (a gap much bigger than a normal grid step), since
# nothing images that gap in "legacy" mode.
GAP_WARN_FACTOR = 5.0
for group in groups:
    for a_idx, b_idx in zip(group.boundary_indices, group.boundary_indices[1:]):
        gap = float(np.linalg.norm(boundary_paths[a_idx][-1] - boundary_paths[b_idx][0]))
        if gap > GAP_WARN_FACTOR * step_size_um:
            print(f"WARNING: tissue {group.tissue} boundaries "
                  f"{boundaries[a_idx].label or '(single)'} -> {boundaries[b_idx].label or '(single)'} "
                  f"are {gap:.0f} um apart ({gap / step_size_um:.1f}x step) in 'legacy' mode -- "
                  f"no transit bridges this gap. Consider "
                  f"TISSUE_PATH_MODE_OVERRIDES[{group.tissue}] = 'transit'.")

# ── Assemble the ordered segment list ────────────────────────────────────
# Each group above becomes one "boundary" segment (its boundaries'
# coordinates concatenated in order, single-element unless merged under
# "legacy"); a dedicated transit bridges every consecutive pair of the
# resulting segments, wrapping the last back to the first so the whole
# acquisition is one cyclic sequence for Dave.
segments = [
    {"kind": "boundary", "tissue": g.tissue, "label": g.label,
     "coords": np.concatenate([boundary_paths[k] for k in g.boundary_indices], axis=0)}
    for g in groups
]

n_top_level = len(segments)
if n_top_level > 1:
    bridged = []
    for k, seg in enumerate(segments):
        bridged.append(seg)
        nxt    = segments[(k + 1) % n_top_level]
        a      = seg["coords"][-1]
        b_next = nxt["coords"][0]
        bridged.append({"kind": "transit", "tissue": seg["tissue"],
                        "label": f"transit_{k + 1}",
                        "coords": create_transit_path(a, b_next, step_size_um, TRANSIT_SPACING)})
    segments = bridged

n_transits = len(segments) - n_top_level
print(f"\n{n_top_level} boundary segment(s) + {n_transits} transit segment(s):")
for seg in segments:
    print(f"  {(seg['label'] or '(single)'):12s} [{seg['kind']:8s}, tissue {seg['tissue']}]: {len(seg['coords'])} FOVs")

In [ ]:
# ── Overview plot: all boundaries, their FOVs, and the transit segments ──
half   = fov_size_um / 2
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(8, 7))
for i, (spec, poly, path) in enumerate(zip(boundaries, boundary_polys, boundary_paths)):
    c = colors[i % len(colors)]
    ax.plot(*poly.exterior.xy, "-", lw=1.2, color=c)
    for x, y in path:
        ax.add_patch(mpatches.Rectangle((x - half, y - half), fov_size_um, fov_size_um,
                                        lw=0.3, edgecolor=c, facecolor=c, alpha=0.15))
    ax.plot(path[:, 0], path[:, 1], "-o", ms=2, lw=0.5, color=c,
            label=f"{spec.label or 'boundary'} ({len(path)})")

for hole in holes:
    ax.fill(*hole.exterior.xy, color="0.8", alpha=0.6)
    ax.plot(*hole.exterior.xy, "k--", lw=0.8)

transit_segments = [seg for seg in segments if seg["kind"] == "transit"]
for k, seg in enumerate(transit_segments):
    tr = seg["coords"]
    ax.plot(tr[:, 0], tr[:, 1], "--x", ms=4, lw=0.8, color="0.35",
            label="transit" if k == 0 else None)

ax.invert_yaxis(); ax.axis("equal")
ax.legend(fontsize=7, loc="best")
plt.title(f"FOV layout ({MODE}) — {SAMPLE_NAME}")
plt.tight_layout()
fig.savefig(POSITIONS_DIR / f"fov_layout_{POSITIONS_TAG}.png", dpi=150)
plt.show()

In [ ]:
# ── Full-sequence path statistics (in acquisition order) ─────────────────
full_path             = np.concatenate([s["coords"] for s in segments], axis=0)
total_um, max_step_um = get_path_stats(full_path)
n_bd_fovs = sum(len(s["coords"]) for s in segments if s["kind"] == "boundary")
n_tr_fovs = sum(len(s["coords"]) for s in segments if s["kind"] == "transit")

print(f"FOVs in acquisition order : {len(full_path)}  "
      f"({n_bd_fovs} boundary + {n_tr_fovs} transit)")
print(f"Full path length          : {total_um / 1000:.1f} mm")
print(f"Max single step           : {max_step_um:.0f} um  "
      f"(transit target ~{TRANSIT_SPACING * step_size_um:.0f} um)")

In [ ]:
# ── Create data/ subfolders for this layout ──────────────────────────────
(DATA_DIR / "mosaic10x").mkdir(parents=True, exist_ok=True)
if MODE == "multi":
    for t in tissues:
        for sub in ("cells", "hybs", "transit"):
            (DATA_DIR / f"tissue_{t}" / sub).mkdir(parents=True, exist_ok=True)
else:  # single / legacy: top-level cells/hybs (+ transit if there are transits)
    for sub in ("cells", "hybs"):
        (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)
    if n_transits:
        (DATA_DIR / "transit").mkdir(parents=True, exist_ok=True)
print(f"Data folders created under {DATA_DIR}")

# ── Write per-segment positions files (referenced by Dave) ───────────────
# Skipped when the label is empty -- that tissue's single/legacy combined
# path, written below as the aggregate positions_{SAMPLE}.txt instead.
written = []
for seg in segments:
    if not seg["label"]:
        continue
    fname = f"positions_{POSITIONS_TAG}_{seg['label']}.txt"
    save_positions_array(seg["coords"], POSITIONS_DIR / fname)
    written.append(fname)

# ── Write per-tissue FOV-only files (boundary FOVs only, no transit) ──────
# A "multi" tissue already collapsed into one "legacy" segment above wrote
# this exact file (positions_{SAMPLE}_T{t}.txt) via the per-segment loop --
# skip it here so it isn't written twice.
if MODE == "multi":
    for t in tissues:
        fname = f"positions_{POSITIONS_TAG}_T{t}.txt"
        if fname in written:
            continue
        coords = np.concatenate(
            [p for spec, p in zip(boundaries, boundary_paths) if spec.tissue == t], axis=0)
        save_positions_array(coords, POSITIONS_DIR / fname)
        written.append(fname)
else:  # single / legacy: one aggregate file of all boundary FOVs
    coords = np.concatenate(boundary_paths, axis=0)
    fname  = f"positions_{POSITIONS_TAG}.txt"
    save_positions_array(coords, POSITIONS_DIR / fname)
    written.append(fname)

print(f"\nWrote {len(written)} positions file(s) to {POSITIONS_DIR}:")
for f in written:
    print(f"  {f}")